# 00 - Dataset Exploration: WLASL

**Objetivo**: Investigar y preparar el dataset WLASL (Word-Level American Sign Language) para entrenar modelos de reconocimiento de palabras ASL.

## Índice
1. Investigación de datasets disponibles
2. Selección y justificación de WLASL
3. Estructura del dataset
4. Descarga y verificación
5. Exploración de los datos
6. Decisiones de preprocesamiento

## 1. Investigación de Datasets Disponibles

| Dataset | Palabras | Videos | Signers | Pros | Contras |
|---------|----------|--------|---------|------|--------|
| **WLASL** | 2000 | ~21K | 100+ | Más citado, subsets flexibles | Videos de YouTube (algunos eliminados) |
| **MS-ASL** | 1000 | ~25K | 200+ | Alta calidad, Microsoft | Requiere solicitud especial |
| **ASL Citizen** | 2700 | ~80K | 50+ | Más reciente, diverso | Muy grande para empezar |
| **How2Sign** | 2000+ | ~35K | 11 | Signing continuo | Muy complejo, pocos signers |

### Decisión: WLASL

**Justificación**:
- Dataset más citado en literatura de reconocimiento de ASL
- Subsets escalables (WLASL100 → WLASL2000)
- Herramientas de descarga disponibles
- Permite comparación con papers existentes

## 2. Estructura del Dataset WLASL

### Formato JSON de anotaciones

```json
{
    "gloss": "hello",           // Palabra/etiqueta
    "instances": [
        {
            "bbox": [x_min, y_min, x_max, y_max],  // Bounding box
            "fps": 25,
            "frame_start": 1,
            "frame_end": -1,     // -1 = hasta el final
            "instance_id": 12345,
            "signer_id": 5,
            "source": "youtube",
            "split": "train",    // train/val/test
            "url": "...",
            "video_id": "abc123"
        }
    ]
}
```

### Subsets disponibles

| Subset | Palabras | Uso recomendado |
|--------|----------|----------------|
| WLASL100 | 100 | Desarrollo y debugging |
| WLASL300 | 300 | Experimentación |
| WLASL1000 | 1000 | Evaluación seria |
| WLASL2000 | 2000 | Evaluación final |

In [2]:
# Configuración inicial
import os
import sys
from pathlib import Path

# Paths del proyecto
PROJECT_ROOT = Path("../../").resolve()
DATASET_DIR = PROJECT_ROOT / "06_datasets" / "wlasl"
RAW_DIR = DATASET_DIR / "raw"
FRAMES_DIR = DATASET_DIR / "frames"
PROCESSED_DIR = DATASET_DIR / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset dir: {DATASET_DIR}")

# Verificar que existen
for d in [RAW_DIR, FRAMES_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"✓ {d.relative_to(PROJECT_ROOT)}")

Project root: /home/fransando/Dev/tfg-sign-language-recognition
Dataset dir: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl
✓ 06_datasets/wlasl/raw
✓ 06_datasets/wlasl/frames
✓ 06_datasets/wlasl/processed


## 3. Descarga del Dataset

### Opción A: Desde GitHub (videos originales)

```bash
# Clonar repositorio
cd 06_datasets/wlasl/raw
git clone https://github.com/dxli94/WLASL.git

# Instalar dependencias
pip install yt-dlp

# Descargar videos
cd WLASL/start_kit
python video_downloader.py

# Preprocesar
python preprocess.py
```

### Opción B: Desde Kaggle (preprocesado)

Dataset ya procesado disponible en:
- https://www.kaggle.com/datasets/risangbaskoro/wlasl-processed

### Opción C: Desde Hugging Face

```python
from datasets import load_dataset
dataset = load_dataset("Voxel51/WLASL")
```

In [3]:
# Primero, descargar el JSON de anotaciones para explorar
import urllib.request
import json

WLASL_JSON_URL = "https://raw.githubusercontent.com/dxli94/WLASL/master/start_kit/WLASL_v0.3.json"
WLASL_JSON_PATH = RAW_DIR / "WLASL_v0.3.json"

if not WLASL_JSON_PATH.exists():
    print("Descargando archivo de anotaciones...")
    urllib.request.urlretrieve(WLASL_JSON_URL, WLASL_JSON_PATH)
    print(f"✓ Descargado: {WLASL_JSON_PATH}")
else:
    print(f"✓ Ya existe: {WLASL_JSON_PATH}")

# Cargar anotaciones
with open(WLASL_JSON_PATH, 'r') as f:
    wlasl_data = json.load(f)

print(f"\nTotal de palabras (glosses): {len(wlasl_data)}")

Descargando archivo de anotaciones...
✓ Descargado: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/raw/WLASL_v0.3.json

Total de palabras (glosses): 2000


In [4]:
# Explorar estructura de los datos
import pandas as pd

# Contar instancias por palabra
word_counts = []
for entry in wlasl_data:
    gloss = entry['gloss']
    n_instances = len(entry['instances'])
    
    # Contar por split
    splits = {'train': 0, 'val': 0, 'test': 0}
    for inst in entry['instances']:
        split = inst.get('split', 'unknown')
        if split in splits:
            splits[split] += 1
    
    word_counts.append({
        'gloss': gloss,
        'total': n_instances,
        'train': splits['train'],
        'val': splits['val'],
        'test': splits['test']
    })

df_words = pd.DataFrame(word_counts)
df_words = df_words.sort_values('total', ascending=False).reset_index(drop=True)

print("Top 20 palabras con más instancias:")
print(df_words.head(20).to_string())

print(f"\n--- Estadísticas generales ---")
print(f"Total palabras: {len(df_words)}")
print(f"Total instancias: {df_words['total'].sum()}")
print(f"Media instancias/palabra: {df_words['total'].mean():.1f}")
print(f"Min instancias: {df_words['total'].min()}")
print(f"Max instancias: {df_words['total'].max()}")

Top 20 palabras con más instancias:
       gloss  total  train  val  test
0       book     40     30    6     4
1      drink     35     25    6     4
2   computer     30     20    5     5
3      chair     26     19    4     3
4     before     26     18    4     4
5         go     26     19    4     3
6        who     25     18    4     3
7    clothes     25     18    4     3
8      candy     24     17    4     3
9       deaf     23     16    4     3
10    cousin     23     16    4     3
11      walk     22     15    4     3
12      thin     22     15    4     3
13        no     22     15    4     3
14      fine     22     15    4     3
15       yes     22     15    4     3
16      year     22     15    4     3
17      help     22     15    4     3
18     woman     21     15    3     3
19    mother     21     15    3     3

--- Estadísticas generales ---
Total palabras: 2000
Total instancias: 21083
Media instancias/palabra: 10.5
Min instancias: 6
Max instancias: 40


In [5]:
# Crear subsets WLASL100, WLASL300, etc.
def get_wlasl_subset(data, top_k):
    """
    Extrae las top_k palabras con más instancias.
    """
    # Ordenar por número de instancias
    sorted_data = sorted(data, key=lambda x: len(x['instances']), reverse=True)
    return sorted_data[:top_k]

# Crear subsets
wlasl100 = get_wlasl_subset(wlasl_data, 100)
wlasl300 = get_wlasl_subset(wlasl_data, 300)

print("WLASL100:")
print(f"  Palabras: {len(wlasl100)}")
print(f"  Total videos: {sum(len(e['instances']) for e in wlasl100)}")
print(f"  Primeras 10 palabras: {[e['gloss'] for e in wlasl100[:10]]}")

print("\nWLASL300:")
print(f"  Palabras: {len(wlasl300)}")
print(f"  Total videos: {sum(len(e['instances']) for e in wlasl300)}")

WLASL100:
  Palabras: 100
  Total videos: 2038
  Primeras 10 palabras: ['book', 'drink', 'computer', 'before', 'chair', 'go', 'clothes', 'who', 'candy', 'cousin']

WLASL300:
  Palabras: 300
  Total videos: 5117


In [6]:
# Explorar una entrada de ejemplo
example = wlasl100[0]

print(f"Palabra: {example['gloss']}")
print(f"Número de instancias: {len(example['instances'])}")
print(f"\nPrimera instancia:")

inst = example['instances'][0]
for key, value in inst.items():
    print(f"  {key}: {value}")

Palabra: book
Número de instancias: 40

Primera instancia:
  bbox: [385, 37, 885, 720]
  fps: 25
  frame_end: -1
  frame_start: 1
  instance_id: 0
  signer_id: 118
  source: aslbrick
  split: train
  url: http://aslbricks.org/New/ASL-Videos/book.mp4
  variation_id: 0
  video_id: 69241


## 4. Decisiones de Preprocesamiento

### Para nuestro pipeline necesitamos:

1. **Extracción de frames**: Convertir videos a secuencias de imágenes
   - FPS: Mantener 25 fps o submuestrear a 10-15 fps
   - Resolución: Redimensionar a 224x224 (estándar para CNNs)

2. **Segmentación temporal**: Usar `frame_start` y `frame_end` para recortar

3. **Normalización**: 
   - Valores de píxel: [0, 1] o ImageNet normalization
   - Secuencias: Padding/truncation a longitud fija o variable

4. **Estructura de salida**:
```
06_datasets/wlasl/processed/
├── wlasl100/
│   ├── train/
│   │   ├── hello/
│   │   │   ├── video_001/
│   │   │   │   ├── frame_000.jpg
│   │   │   │   ├── frame_001.jpg
│   │   │   │   └── ...
│   │   │   └── video_002/
│   │   └── goodbye/
│   ├── val/
│   └── test/
└── metadata.json
```

In [7]:
# Guardar WLASL100 subset para uso posterior
wlasl100_path = RAW_DIR / "wlasl100_subset.json"

with open(wlasl100_path, 'w') as f:
    json.dump(wlasl100, f, indent=2)

print(f"✓ Guardado subset WLASL100: {wlasl100_path}")

# También guardar lista de palabras para referencia
words_list_path = RAW_DIR / "wlasl100_words.txt"
with open(words_list_path, 'w') as f:
    for entry in wlasl100:
        f.write(f"{entry['gloss']}\n")

print(f"✓ Guardada lista de palabras: {words_list_path}")

✓ Guardado subset WLASL100: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/raw/wlasl100_subset.json
✓ Guardada lista de palabras: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/raw/wlasl100_words.txt


## 5. Próximos Pasos

1. **Notebook 01**: Implementar descarga y extracción de frames
2. **Notebook 02**: Crear DataLoader para secuencias
3. **Notebook 03**: Entrenar modelo CNN+LSTM baseline

---

### Referencias

- [WLASL GitHub](https://github.com/dxli94/WLASL)
- [Paper: Word-level Deep Sign Language Recognition from Video](https://arxiv.org/abs/1910.11006)
- [WLASL en Kaggle](https://www.kaggle.com/datasets/risangbaskoro/wlasl-processed)